In [0]:
# Cell 1 — read silver features table
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("raw.silver.cms_claims_features")

print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")
print("\nSample:")
df.select("provider_id", "specialty", "state",
          "avg_submitted_charge", "avg_medicare_paid",
          "charge_to_payment_ratio", "provider_charge_zscore").show(3, truncate=False)

Rows: 1,999,992
Columns: 37

Sample:
+-----------+------------------------------------------------+-----+--------------------+-----------------+-----------------------+----------------------+
|provider_id|specialty                                       |state|avg_submitted_charge|avg_medicare_paid|charge_to_payment_ratio|provider_charge_zscore|
+-----------+------------------------------------------------+-----+--------------------+-----------------+-----------------------+----------------------+
|1003041468 |Advanced Heart Failure and Transplant Cardiology|WA   |185.0               |53.701782178     |3.444950847009113      |-0.36091983516276105  |
|1003041468 |Advanced Heart Failure and Transplant Cardiology|WA   |66.0                |19.680571429     |3.353561162494841      |-0.648076687337775    |
|1003041468 |Advanced Heart Failure and Transplant Cardiology|WA   |19.0                |5.4576           |3.4813837584286134     |-0.7614915785329486   |
+-----------+--------------------

In [0]:
# Cell 2 — create gold schema and build provider risk table
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Create gold schema
spark.sql("CREATE SCHEMA IF NOT EXISTS raw.gold")
spark.sql("SHOW SCHEMAS IN raw").show()

+------------------+
|      databaseName|
+------------------+
|           default|
|              gold|
|information_schema|
|       jaffle_shop|
|            silver|
|            stripe|
+------------------+



In [0]:
# Cell 3 — aggregate to provider level (Gold)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w_specialty = Window.partitionBy("specialty")

df_gold = df \
    .groupBy("provider_id", "specialty", "state", "city", "zip_code", "entity_code") \
    .agg(
        F.count("*").alias("total_procedures"),
        F.sum("total_beneficiaries").alias("total_beneficiaries"),
        F.sum("total_services").alias("total_services"),
        F.round(F.sum(F.col("avg_submitted_charge") * F.col("total_services")), 2).alias("total_billed"),
        F.round(F.sum(F.col("avg_medicare_paid") * F.col("total_services")), 2).alias("total_paid"),
        F.round(F.avg("charge_to_payment_ratio"), 4).alias("avg_charge_ratio"),
        F.round(F.avg("billing_intensity"), 4).alias("avg_billing_intensity"),
        F.round(F.max("provider_charge_zscore"), 4).alias("max_zscore"),
        F.round(F.avg("provider_charge_zscore"), 4).alias("avg_zscore"),
        F.countDistinct("procedure_code").alias("unique_procedures"),
        F.round(F.avg("facility_flag"), 4).alias("facility_rate")
    ) \
    .withColumn("specialty_rank",
        F.rank().over(Window.partitionBy("specialty").orderBy(F.desc("max_zscore")))) \
    .withColumn("risk_tier",
        F.when(F.col("max_zscore") >= 3, "Critical")
         .when(F.col("max_zscore") >= 1, "Elevated")
         .otherwise("Normal")) \
    .withColumn("gold_processed_at", F.current_timestamp())

print(f"Gold rows (unique providers): {df_gold.count():,}")
print("\nRisk tier distribution:")
df_gold.groupBy("risk_tier").count().orderBy("count").show()
print("\nTop 5 Critical providers:")
df_gold.filter(F.col("risk_tier") == "Critical") \
    .orderBy(F.desc("max_zscore")) \
    .select("provider_id", "specialty", "state", "max_zscore", "avg_charge_ratio", "total_billed") \
    .show(5, truncate=False)

Gold rows (unique providers): 236,681

Risk tier distribution:
+---------+------+
|risk_tier| count|
+---------+------+
| Critical| 17891|
| Elevated| 51910|
|   Normal|166880|
+---------+------+


Top 5 Critical providers:
+-----------+--------------------+-----+----------+----------------+------------+
|provider_id|specialty           |state|max_zscore|avg_charge_ratio|total_billed|
+-----------+--------------------+-----+----------+----------------+------------+
|1013172907 |Internal Medicine   |MD   |119.8422  |3.9583          |5372501.0   |
|1104900448 |Internal Medicine   |MD   |88.2472   |4.1828          |2951809.0   |
|1073824892 |Diagnostic Radiology|CA   |88.1044   |23.6825         |3081910.0   |
|1073778429 |Cardiology          |NJ   |80.4744   |74.5557         |1.7821284E7 |
|1144287574 |Cardiology          |KS   |79.8454   |8.4083          |7493829.8   |
+-----------+--------------------+-----+----------+----------------+------------+
only showing top 5 rows


In [0]:
# Cell 4 — write Gold provider risk table
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("raw.gold.provider_risk")

print(f"Gold table written: {df_gold.count():,} providers")
print("\nSchema:")
df_gold.printSchema()

Gold table written: 236,681 providers

Schema:
root
 |-- provider_id: long (nullable = true)
 |-- specialty: string (nullable = true)
 |-- state: string (nullable = true)
 |-- city: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- entity_code: string (nullable = true)
 |-- total_procedures: long (nullable = false)
 |-- total_beneficiaries: long (nullable = true)
 |-- total_services: double (nullable = true)
 |-- total_billed: double (nullable = true)
 |-- total_paid: double (nullable = true)
 |-- avg_charge_ratio: double (nullable = true)
 |-- avg_billing_intensity: double (nullable = true)
 |-- max_zscore: double (nullable = true)
 |-- avg_zscore: double (nullable = true)
 |-- unique_procedures: long (nullable = false)
 |-- facility_rate: double (nullable = true)
 |-- specialty_rank: integer (nullable = false)
 |-- risk_tier: string (nullable = false)
 |-- gold_processed_at: timestamp (nullable = false)



In [0]:
# Cell 5 — verify Gold table
df_check = spark.table("raw.gold.provider_risk")

print(f"Rows: {df_check.count():,}")
print(f"Columns: {len(df_check.columns)}")

print("\nRisk tier summary:")
df_check.groupBy("risk_tier") \
    .agg(
        F.count("*").alias("providers"),
        F.round(F.sum("total_billed"), 2).alias("total_billed_usd")
    ) \
    .orderBy("providers") \
    .show()

print("\nTables in raw.gold:")
spark.sql("SHOW TABLES IN raw.gold").show()

Rows: 236,681
Columns: 20

Risk tier summary:
+---------+---------+-----------------+
|risk_tier|providers| total_billed_usd|
+---------+---------+-----------------+
| Critical|    17891| 2.04239637627E10|
| Elevated|    51910|2.312210126846E10|
|   Normal|   166880|2.711439975903E10|
+---------+---------+-----------------+


Tables in raw.gold:
+--------+-------------+-----------+
|database|    tableName|isTemporary|
+--------+-------------+-----------+
|    gold|provider_risk|      false|
+--------+-------------+-----------+

